In [5]:
import pandas as pd
import pyomo.environ as pyo
from itertools import product
from pyomo.environ import value
import numpy as np
import random
import math

model = pyo.ConcreteModel()


# 1. SETS

T=[0, 1, 2, 3, 4, 5]  # horizon temporel

#définition des zones
dA = pd.read_csv("AREAS.csv", sep=";")  
na  = 15                                        #nombre de zones de la petite instance
n = len(dA["AREAS"])                            #nombre de zones au total
A = [dA["AREAS"][i] for i in random.sample(range(0, n), na) ]



I=['ORANGE','FREE MOBILE', 'BOUYGUES TELECOM','SFR']  # opérateurs
τ='ORANGE'  # notre opérateur
O=['o3G-FREE MOBILE', 'o3G-BOUYGUES TELECOM', 'o3G-SFR','o3G-ORANGE','o4G-FREE MOBILE','o4G-BOUYGUES TELECOM','o4G-SFR','o4G-ORANGE','o5G-FREE MOBILE','o5G-BOUYGUES TELECOM','o5G-SFR','o5G-ORANGE']  
O_i = {"ORANGE" : ['o3G-ORANGE' , 'o4G-ORANGE', 'o5G-ORANGE'] , 'FREE MOBILE' : ['o3G-FREE MOBILE', 'o4G-FREE MOBILE', 'o5G-FREE MOBILE'] ,
      'BOUYGUES TELECOM' : ['o3G-BOUYGUES TELECOM', 'o4G-BOUYGUES TELECOM', 'o5G-BOUYGUES TELECOM'] , 'SFR' : ['o3G-SFR', 'o4G-SFR', 'o5G-SFR']  }# offres

NG = 'o5G-ORANGE'

model.T = pyo.Set(initialize=T)  # horizon temporel
model.A = pyo.Set(initialize=A)  # zones
model.I = pyo.Set(initialize=I)  # opérateurs
model.O = pyo.Set(initialize=O)  # offres !!! il faut mettre l'offre de chaque opérateur
model.O_i = O_i




In [6]:
### Couples utiles
C_space = list(product([0,1], repeat=len(I)))  # Toutes les combinaisons de couverture
df = pd.read_csv("AREAS_SITES_LINK.csv", sep=";")   # contient colonnes "a" et "s"

Sa_dict = {}
S = [] 

i = 0
for _, row in df.iterrows():
    zone = row["AREAS"]
    site = row["SITES"]
    if zone not in A:             #on ne considère que les zones sélectionnées dans A
        continue

    if zone not in Sa_dict :
        Sa_dict[zone] = []

    if site not in Sa_dict[zone]:     #éviter d'ajouter plusieurs fois la même zone
        Sa_dict[zone].append(site)
        
    if site not in S:
        S.append(site)

model.S = pyo.Set(initialize=S) # sites de l’opérateur i

As_dict = {}

for _, row in df.iterrows():
    zone = row["AREAS"]
    site = row["SITES"]
    if site not in S:
        continue

    if site not in As_dict:
        As_dict[site] = []
    if zone in A and zone not in As_dict[site]:
        As_dict[site].append(zone)



def _C_tuple(m, t, a, r_tau_val):
    C_list = [r_tau_val]
    autres_operateurs = list(m.I)[1:] # On respecte ton indexation (tau est à l'index 0)
    for i in autres_operateurs:
        C_list.append(pyo.value(m.Rcomp[t, a, i]))
    return tuple(C_list)

    


model.Cvec = pyo.Set(initialize=C_space)   # Toutes les combinaisons de couverture
model.Sa = Sa_dict                         # mapping a → {sites}
model.As = As_dict                         # mapping s → {zones}

# 2. PARAMÈTRES
df_zmax = pd.read_csv("OPERATIONAL_LIMITS.csv", sep=";")
Zmax_data = [0, 5, 5, 5, 5, 5]                            #modifiable
model.Zmax = pyo.Param(model.T, initialize = Zmax_data)  # nombre max de sites déployables par période

df_qa = pd.read_csv("STRATEGIC_GUIDELINES.csv", sep=";")
QA_data = {row.TIME_SLOTS: row["QA"] for _,row in df_qa.iterrows()}
QA_data[0] = 0
model.QA = pyo.Param(model.T, initialize = QA_data) # couverture minimale de la population par période

df = pd.read_csv("AREAS.csv", sep=";")

ua0_data = {}
for _, row in df.iterrows():
    for col in df.columns [1:]:
        if row["AREAS"] in A :                      #on ne prend que les aires de la petite instance
            ua0_data[row["AREAS"],col.split("-")[1], col] = row[col]    #partie entière retirée
for a in A:
    for i in I:
        for o in O:
            if (a,i,o) not in ua0_data:
                ua0_data[a,i,o] = 0

    
        
model.ua0 = pyo.Param(model.A, model.I, model.O, initialize=ua0_data)  # utilisateurs initiaux

df_dng = pd.read_csv("DEMAND.csv", sep=";")

DNG_data = {row.TIME_SLOTS: row["5G"] for _,row in df_dng.iterrows()}
DNG_data[0] = 0
model.DNG = pyo.Param(model.T, initialize = DNG_data)  # DNG dépend du temps

df_capang = pd.read_csv("CAPACITY.csv", sep=";")
CAPANG_data = {row.TIME_SLOTS: row["5G"] for _,row in df_capang.iterrows()}
CAPANG_data[0] = 0
model.CAPANG = pyo.Param(model.T, initialize = CAPANG_data) # DNG et CAPANG dépendent du temps

df_ua = pd.read_csv("AREAS.csv", sep=";")
u_a_data = {}
for _, row in df.iterrows():
    if row["AREAS"] in A :
        somme =0
        for col in df.columns [1:]:
            somme += row[col]
        u_a_data [row["AREAS"]]= somme

model.u_a = pyo.Param(model.A, initialize = u_a_data)  # utilisateurs totaux dans la zone a

df_Rcomp = pd.read_csv("COMPETITORS_STRATEGY.csv", sep=";")
Rcomp_data = {}
for _, row in df_Rcomp.iterrows():
    if row["AREAS"] in A :
        for col in df_Rcomp.columns[2:5]:
            Rcomp_data[row["TIME_SLOTS"],row["AREAS"], col] = 1 * row[col]
model.Rcomp = pyo.Param(model.T, model.A, model.I, initialize=Rcomp_data)  # couverture des autres opérateurs

#f

df_fdata = pd.read_csv("UPGRADE_FUNCTION.csv", sep=";")

f_data = {}
for a in A:
    for _, row in df_fdata.iterrows():
        C  = (row["ORANGE"], row["FREE MOBILE"], row["BOUYGUES TELECOM"], row["SFR"])
        O1 = row["OFFERS"] + "-" + row["FROM_OPERATOR"]
        O2 = "o5G-" + row["TO_OPERATOR"]
        f_data[(a, C, O1, O2)] = row["PERCENTAGES"]

O_full= ['o3G-FREE MOBILE', 'o3G-BOUYGUES TELECOM', 'o3G-SFR', 'o3G-ORANGE',
       'o4G-FREE MOBILE', 'o4G-BOUYGUES TELECOM', 'o4G-SFR', 'o4G-ORANGE',
          'o5G-FREE MOBILE', 'o5G-BOUYGUES TELECOM', 'o5G-SFR', 'o5G-ORANGE']

for a in A:
    for C in C_space:  # tuples de 0/1
        for O1 in O_full:    # toutes les offres existantes
            for O2 in O_full:                                               
                if (a, C, O1, O2) not in f_data:
                   f_data[a, C, O1, O2] = 0

for a in A:
    for C in C_space:  
        for O1 in O_full:
            v = 1
            for O2 in O_full:
                if O2 != O1 :
                    v -= f_data[a, C, O1, O2]
            f_data[a, C, O1, O1] = v                             #tous ceux qui ne bougent pas restent chez eux.



model.f = pyo.Param(model.A, model.Cvec, model.O, model.O, initialize=f_data) # taux de migration

#un grand M pour les contraintes de Big-M
M_data = {}
eps = 1e-4
for a in model.A:
    
    for o in model.O:
        for t in model.T:
            max_diff = 0.0
            for i in model.I:
                for o_prev in model.O_i[i]:
                    diff = abs(
                        pyo.value(model.f[a, _C_tuple(model, t, a, 1), o_prev, o]) -
                        pyo.value(model.f[a, _C_tuple(model, t , a, 0), o_prev, o])
                    )
                    max_diff = max(max_diff, diff)

            M_data[(a, o, t)] = max_diff * pyo.value(model.u_a[a]) + eps

model.M = pyo.Param(model.A, model.O, model.T, initialize=M_data, mutable=True)

In [7]:
# 3. VARIABLES

model.z = pyo.Var(model.T, model.S, within=pyo.Binary)
model.r = pyo.Var(model.T, model.A, within=pyo.Binary)

# Variable de linéarisation S (16)
model.S_var = pyo.Var(model.T, model.A, model.I, model.O, within=pyo.Reals)

model.u = pyo.Var(model.T, model.A, model.I, model.O, within=pyo.NonNegativeReals)
model.u_site = pyo.Var(model.T, model.A, model.S, within=pyo.NonNegativeReals)  # on ne définit pas l'opérateur et l'offre car on ne parle que de notre opérateur et la NG

In [8]:
# 4. CONTRAINTES

# (2) r_ta ≤ sum_s z_ts
def coverage_upper(m, t, a):
    return m.r[t, a] <= sum(m.z[t, s] for s in Sa_dict[a])
model.c_2 = pyo.Constraint(model.T, model.A, rule=coverage_upper)

# (3) z_ts ≤ r_ta  ∀ s couvrant a
def coverage_lower(m, t, s, a):
    if a in As_dict[s]:
        return m.z[t, s] <= m.r[t, a]
    return pyo.Constraint.Skip
model.c_3 = pyo.Constraint(model.T, model.S, model.A, rule=coverage_lower)


# (4) Évolution des utilisateurs
def u_eq_rule(m, t, a, i, o):
    if o in O_i[i]:
        if t == min(m.T):
            return m.u[t, a, i, o] == m.ua0[a, i, o]
        
        C0 = _C_tuple(m, t, a, 0) # Couverture si on ne déploie PAS (r=0)
        
        # Somme des migrations basées sur le taux f0
        migration_f0 = sum(m.f[a, C0, o_prev, o] * m.u[t-1, a, i_prev, o_prev]
                        for i_prev in m.I for o_prev in m.O_i[i_prev])
                        
        return m.u[t, a, i, o] == m.S_var[t, a, i, o] + migration_f0
    else:
        return pyo.Constraint.Skip
model.c_4 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=u_eq_rule)

# (5) Expression de sigma (pour la linéarisation de S)
def sigma_rule(m, t, a, i, o):
    if t == min(m.T):
        return 0.0
    C0 = _C_tuple(m, t, a, 0) # r = 0
    C1 = _C_tuple(m, t, a, 1) # r = 1
    
    return sum((m.f[a, C1, o_prev, o] - m.f[a, C0, o_prev, o]) * m.u[t-1, a, i_prev, o_prev]
               for i_prev in m.I for o_prev in m.O_i[i_prev])
model.sigma = pyo.Expression(model.T, model.A, model.I, model.O, rule=sigma_rule)

# (6) S <= sigma + (1-r) * M
def c_6_rule(m, t, a, i, o):
    if t == min(m.T) or o not in O_i[i]: return pyo.Constraint.Skip
    M = m.M[a, o , t]  # Utilisation du M calculé pour cette combinaison spécifique
    return m.S_var[t, a, i, o] <= m.sigma[t, a, i, o] + (1 - m.r[t, a]) * M
model.c_6 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=c_6_rule)

# (7) S <= r * M 
def c_7_rule(m, t, a, i, o):
    if t == min(m.T) or o not in O_i[i]: return m.S_var[t, a, i, o] == 0.0
    M = m.M[a, o , t]  # Utilisation du M calculé pour cette combinaison spécifique
    return m.S_var[t, a, i, o] <= m.r[t, a] * M
model.c_7 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=c_7_rule)

# (8) S >= sigma - (1-r) * M
def c_8_rule(m, t, a, i, o):
    if t == min(m.T) or o not in O_i[i]: return pyo.Constraint.Skip
    M = m.M[a, o , t]  # Utilisation du M calculé pour cette combinaison spécifique
    return m.S_var[t, a, i, o] >= m.sigma[t, a, i, o] -  (1 - m.r[t, a]) * M
model.c_8 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=c_8_rule)

# (9) S >= -r * M
def c_9_rule(m, t, a, i, o):
    if t == min(m.T) or o not in O_i[i]: return pyo.Constraint.Skip
    M = m.M[a, o , t]  # Utilisation du M calculé pour cette combinaison spécifique
    return m.S_var[t, a, i, o] >= -m.r[t, a] * M
model.c_9 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=c_9_rule)

# (10) u_NO = somme sur les sites
def assign_users(m, t, a):
    return m.u[t, a, τ, NG] == sum(m.u_site[t, a, s] for s in Sa_dict[a])
model.c_10 = pyo.Constraint(model.T, model.A, rule=assign_users)


# (11) Capacité du réseau
def capacity(m, t, s):
    return sum(model.DNG[t] * m.u_site[t, a, s] for a in As_dict[s]) <= model.CAPANG[t] * m.z[t, s]
model.c_11 = pyo.Constraint(model.T, model.S, rule=capacity)

# (8) budget sur le nombre de sites déployés par période
def limit_z(m, t):
    if t == min(m.T):
        return sum(m.z[t, s] for s in model.S) <= model.Zmax[t]
    return sum(m.z[t, s] - m.z[t-1, s] for s in model.S) <= model.Zmax[t]

model.c_8 = pyo.Constraint(model.T, rule=limit_z)

# (12) Budget sur le nombre de sites déployés par période
def limit_z(m, t):
    if t == min(m.T):
        return sum(m.z[t, s] for s in m.S) <= m.Zmax[t]
    return sum(m.z[t, s] - m.z[t-1, s] for s in m.S) <= m.Zmax[t]
model.c_12 = pyo.Constraint(model.T, rule=limit_z)

# (13) Couverture population minimale
def cov_pop(m, t):
    return sum(m.u_a[a] * m.r[t, a] for a in m.A) >= m.QA[t] * sum(m.u_a[a] for a in m.A)
model.c_13 = pyo.Constraint(model.T, rule=cov_pop)

# La contrainte de la crossance des z_st
def growth_z(m, t, s):
    if t == min(m.T):
        return pyo.Constraint.Skip
    return m.z[t, s]>= m.z[t-1, s]
model.c_growth_z = pyo.Constraint(model.T, model.S, rule=growth_z)

'pyomo.core.base.constraint.IndexedConstraint'>) on block unknown with a new
Component (type=<class 'pyomo.core.base.constraint.IndexedConstraint'>). This
is usually indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().


In [9]:
# 5. OBJECTIF

def objective(m):
    T_end = max(m.T)
    return sum(m.u[T_end, a, τ, NG] for a in m.A)
model.obj = pyo.Objective(rule=objective, sense=pyo.maximize)


# Create solver
solver = pyo.SolverFactory('glpk')

# Solve
results = solver.solve(model, tee=True)

# Display solver status
print("\nSolver status:", results.solver.status)
print("Termination condition:", results.solver.termination_condition)

# 1) Valeur de l'objectif
print("Valeur de l'objectif :", value(model.obj))

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /tmp/tmpf8a58htk.glpk.raw --wglp /tmp/tmpg0chfpw6.glpk.glp --cpxlp
 /tmp/tmpamxfhr_6.pyomo.lp
Reading problem data from '/tmp/tmpamxfhr_6.pyomo.lp'...
/tmp/tmpamxfhr_6.pyomo.lp:45043: warning: lower bound of variable 'x18' redefined
/tmp/tmpamxfhr_6.pyomo.lp:45043: warning: upper bound of variable 'x18' redefined
7659 rows, 5677 columns, 16349 non-zeros
180 integer variables, all of which are binary
45223 lines were read
Writing problem data to '/tmp/tmpg0chfpw6.glpk.glp'...
38389 lines were written
GLPK Integer Optimizer 5.0
7659 rows, 5677 columns, 16349 non-zeros
180 integer variables, all of which are binary
Preprocessing...
5 hidden covering inequaliti(es) were detected
389 constraint coefficient(s) were reduced
3891 rows, 1833 columns, 11375 non-zeros
150 integer variables, all of which are binary
Scaling...
 A: min|aij| =  5.551e-17  max|aij| =  3.009e+03  ratio =  5.421e+19
GM: min|aij| =  1.014

KeyboardInterrupt: 

| Modèle | Nombre de contraintes | Nombre de variables | Valeur de la fonction objective | Temps solveur (GLPK) | 
|--------|------------------------|---------------------|----------------------------------|----------------------|
| Premier modèle | 39440 | 16297 | 3731.042653190765 | 4.2 s |  
| Second modèle  | 3020 | 1412  | 3731.042653190766 | 0.5s | 